# GlassBox AutoML Demo

This notebook demonstrates the transparent end-to-end workflow: inspect data, preprocess it, train several GlassBox models through AutoFit, and inspect the JSON report.

In [1]:
from pathlib import Path
import csv
import json
import sys

import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "data" / "sample.csv"
DATA_PATH

PosixPath('/home/gyro/Desktop/glassbox/data/sample.csv')

## 1. Load The Sample CSV

In [2]:
with DATA_PATH.open("r", encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))

rows[:3], len(rows)

([{'age': '22',
   'income': '32000',
   'visits': '3',
   'plan': 'basic',
   'region': 'north',
   'is_student': '1',
   'purchased': '0'},
  {'age': '25',
   'income': '41000',
   'visits': '5',
   'plan': 'basic',
   'region': 'east',
   'is_student': '0',
   'purchased': '0'},
  {'age': '28',
   'income': '52000',
   'visits': '7',
   'plan': 'plus',
   'region': 'east',
   'is_student': '0',
   'purchased': '1'}],
 25)

## 2. EDA: Column Types, Profiles, Correlations, Outliers

In [3]:
from glassbox.eda import infer_column_types, profile_numeric_columns, build_pearson_correlation_matrix, iqr_outlier_handler

columns = {name: [row[name] for row in rows] for name in rows[0]}
typed_columns = {
    "age": np.array([float(value) for value in columns["age"]]),
    "income": np.array([np.nan if value == "" else float(value) for value in columns["income"]]),
    "visits": np.array([float(value) for value in columns["visits"]]),
    "plan": np.array(columns["plan"], dtype=object),
    "region": np.array(columns["region"], dtype=object),
    "is_student": np.array([int(value) for value in columns["is_student"]]),
    "purchased": np.array([int(value) for value in columns["purchased"]]),
}

infer_column_types(typed_columns)

{'age': 'numerical',
 'income': 'numerical',
 'visits': 'numerical',
 'plan': 'categorical',
 'region': 'categorical',
 'is_student': 'boolean',
 'purchased': 'boolean'}

In [4]:
numeric_names = ["age", "income", "visits"]
numeric = np.column_stack([typed_columns[name] for name in numeric_names])

income_mean = np.nanmean(numeric[:, 1])
numeric[np.isnan(numeric[:, 1]), 1] = income_mean

profile = profile_numeric_columns(numeric, numeric_names)
corr, corr_names, high_corr = build_pearson_correlation_matrix(numeric, numeric_names)
outlier_rows = iqr_outlier_handler(numeric, mode="flag", return_row_indices=True)

profile, high_corr, outlier_rows

({'age': {'mean': 36.52,
   'median': 36.0,
   'mode': 22.0,
   'std': 9.282758210790583,
   'variance': 86.16959999999999,
   'skewness': 0.22757436715913534,
   'kurtosis': -0.9504480713009822},
  'income': {'mean': 70500.0,
   'median': 66000.0,
   'mode': 32000.0,
   'std': 33225.29157133162,
   'variance': 1103920000.0,
   'skewness': 2.800662552966179,
   'kurtosis': 9.654077264510734},
  'visits': {'mean': 7.64,
   'median': 7.0,
   'mode': 5.0,
   'std': 3.474248120097354,
   'variance': 12.0704,
   'skewness': 0.400678712610743,
   'kurtosis': -0.6897343291923854}},
 [('age', 'visits', 0.8628427210260551)],
 [24])

## 3. Preprocessing Pipeline

In [5]:
from glassbox.preprocessing import SimpleImputer, StandardScaler, OneHotEncoder

categorical = np.array(
    [[row["plan"], row["region"], row["is_student"]] for row in rows],
    dtype=object,
)

numeric_clean = SimpleImputer(strategy="mean").fit_transform(numeric)
numeric_scaled = StandardScaler().fit_transform(numeric_clean)
categorical_clean = SimpleImputer(strategy="mode").fit_transform(categorical)
categorical_encoded = OneHotEncoder().fit_transform(categorical_clean)
X_processed = np.hstack([numeric_scaled, categorical_encoded.astype(float)])

X_processed.shape

(25, 12)

## 4. AutoFit: Full Model Selection

In [6]:
from glassbox.agent import auto_fit

report = auto_fit(str(DATA_PATH), target_column="purchased", task="auto", search="random", time_budget=20)

print(json.dumps({
    "task": report["task"],
    "best_model": report["best_model"],
    "cv_score": report["cv_score"],
    "best_params": report["best_params"],
    "candidate_models": report["candidate_models"],
    "eda_overview": report["eda_summary"]["overview"],
    "top_features": report["top_features"][:5],
}, indent=2))

{
  "task": "classification",
  "best_model": "RandomForestClassifier",
  "cv_score": 0.96,
  "best_params": {
    "n_estimators": 100,
    "max_depth": null,
    "min_samples_split": 2
  },
  "candidate_models": [
    "LogisticRegression",
    "DecisionTreeClassifier",
    "RandomForestClassifier",
    "GaussianNaiveBayes",
    "KNearestNeighbors_clf"
  ],
  "eda_overview": {
    "feature_count": 6,
    "type_counts": {
      "numerical": 3,
      "categorical": 2,
      "boolean": 1
    },
    "target_type": "boolean",
    "outlier_counts": {
      "age": 0,
      "income": 1,
      "visits": 0
    },
    "high_collinearity_count": 3
  },
  "top_features": [
    {
      "feature": "visits",
      "importance": 0.3373159132042193
    },
    {
      "feature": "plan=basic",
      "importance": 0.28739598434393576
    },
    {
      "feature": "age",
      "importance": 0.15689806741327403
    },
    {
      "feature": "income",
      "importance": 0.07243205866154409
    },
    {
     

## 5. Search Leaderboard

In [7]:
report["search_results"][:10]

[{'model': 'RandomForestClassifier',
  'score': 0.96,
  'params': {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2}},
 {'model': 'RandomForestClassifier',
  'score': 0.96,
  'params': {'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 2}},
 {'model': 'LogisticRegression',
  'score': 0.9199999999999999,
  'params': {'learning_rate': 0.1,
   'n_iterations': 1500,
   'tol': 1e-06,
   'threshold': 0.6}},
 {'model': 'LogisticRegression',
  'score': 0.9199999999999999,
  'params': {'learning_rate': 0.1,
   'n_iterations': 800,
   'tol': 1e-06,
   'threshold': 0.6}},
 {'model': 'LogisticRegression',
  'score': 0.9199999999999999,
  'params': {'learning_rate': 0.01,
   'n_iterations': 800,
   'tol': 1e-07,
   'threshold': 0.5}},
 {'model': 'LogisticRegression',
  'score': 0.9199999999999999,
  'params': {'learning_rate': 0.1,
   'n_iterations': 300,
   'tol': 1e-07,
   'threshold': 0.6}},
 {'model': 'LogisticRegression',
  'score': 0.9199999999999999,
  'params': {'learn